# Baseline and psychometric models

Executable scientific definitions and computed results are presented below. Data, fitted models, tables and figures are stored in the corresponding standard project directories. Earlier experiments are preserved separately in `Data/legacy/Notebooks/` and are not mixed with the current results.

In [1]:
from pathlib import Path
import sys, types, hashlib, importlib.abc, importlib.util
import nbformat
import pandas as pd
from IPython.core.magic import register_cell_magic
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Notebooks').is_dir() and (p/'Data').is_dir())
MODULE_NOTEBOOKS={'revision_data': '02_data_preprocessing_feature_engineering_and_splits.ipynb', 'revision_models': '03_baseline_models.ipynb', 'revision_sensitivity': '02_data_preprocessing_feature_engineering_and_splits.ipynb', 'revision_neural': '04_sota_models.ipynb', 'revision_evaluation': '06_final_validation_tables_figures_and_reports.ipynb', 'revision_supplemental': '06_final_validation_tables_figures_and_reports.ipynb', 'revision_provenance': '05_proposed_hybrid_model_and_ablations.ipynb', 'revision_validation': '02_data_preprocessing_feature_engineering_and_splits.ipynb', 'revision_closure': '06_final_validation_tables_figures_and_reports.ipynb', 'revision_status': '06_final_validation_tables_figures_and_reports.ipynb'}

class NotebookSourceLoader(importlib.abc.Loader):
    def create_module(self,spec):return None
    def exec_module(self,module):
        path=ROOT/'Notebooks'/MODULE_NOTEBOOKS[module.__name__]
        notebook=nbformat.read(path,4)
        cell=next(c for c in notebook.cells if c.metadata.get('research_module')==module.__name__)
        source=cell.source.split('\n',1)[1]
        module.__file__=str(path)
        module.__notebook_source_sha256__=hashlib.sha256(source.encode()).hexdigest()
        exec(compile(source,str(path)+'#'+cell.id,'exec'),module.__dict__)

class NotebookSourceFinder(importlib.abc.MetaPathFinder):
    def find_spec(self,fullname,path=None,target=None):
        if fullname in MODULE_NOTEBOOKS:
            return importlib.util.spec_from_loader(fullname,NotebookSourceLoader())
sys.meta_path=[f for f in sys.meta_path if type(f).__name__!='NotebookSourceFinder']
sys.meta_path.insert(0,NotebookSourceFinder())

@register_cell_magic
def research_module(line,source):
    """Publish the visible functions for reuse by other notebooks; no hidden helper scripts."""
    name=line.strip();digest=hashlib.sha256(source.rstrip('\n').encode()).hexdigest()
    existing=sys.modules.get(name)
    if existing is not None and existing.__notebook_source_sha256__==digest:return
    module=types.ModuleType(name);module.__file__=str(ROOT/'Notebooks'/MODULE_NOTEBOOKS[name])
    module.__notebook_source_sha256__=digest;sys.modules[name]=module
    exec(compile(source.rstrip('\n'),module.__file__,'exec'),module.__dict__)

@register_cell_magic
def legacy_snapshot(line,cell):
    """Archived analysis is preserved but is not part of the current execution."""
    return None

import revision_data as rd
artifact_path=rd.artifact_path
import matplotlib as mpl
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('png')
mpl.rcParams.update({'figure.dpi':350,'savefig.dpi':350})
pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',100)

### Models — executable definitions

The functions below are the source used by this notebook and reused by the other notebooks. Defining them does not repeat model fitting.

In [2]:
%%research_module revision_models
"""Shared revised metrics, HGB anchors, residual memories and temporal fitting."""
from revision_data import artifact_path, code_digest, source_matches, artifact_matches, fit_contract_matches
from pathlib import Path
import ast
import copy
import hashlib
import json
import time
import joblib
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits
from revision_data import ROOT,V,KEYS,FEATURES,protocol,digest,object_hash,save_json,register,table

def safe_p(p): return np.clip(np.asarray(p,float),1e-6,1-1e-6)

def calibration_coefficients(y,p,w):
    if np.unique(y[w>0]).size<2 or np.std(logit(p))<1e-12: return np.nan,np.nan
    fit=LogisticRegression(C=1e6,max_iter=300).fit(logit(p).reshape(-1,1),y,sample_weight=w)
    return float(fit.intercept_[0]),float(fit.coef_[0,0])

def metrics(y,p,weights=None,bins=15,equal_frequency=False):
    y=np.asarray(y,int);p=safe_p(p);w=np.ones(len(y)) if weights is None else np.asarray(weights,float)
    w=w/w.sum();both=np.unique(y[w>0]).size==2
    edges=np.unique(np.quantile(p,np.linspace(0,1,bins+1))) if equal_frequency else np.linspace(0,1,bins+1)
    ids=np.clip(np.searchsorted(edges,p,side='right')-1,0,max(0,len(edges)-2));ece=0.
    for b in np.unique(ids):
        m=ids==b;mass=w[m].sum()
        if mass: ece+=abs(np.dot(w[m],y[m]-p[m]))
    intercept,slope=calibration_coefficients(y,p,w*len(y))
    return {'ROC_AUC':roc_auc_score(y,p,sample_weight=w) if both else np.nan,
        'AP_correct':average_precision_score(y,p,sample_weight=w) if both else np.nan,
        'AP_incorrect':average_precision_score(1-y,1-p,sample_weight=w) if both else np.nan,
        'Log_Loss':float(np.dot(w,-y*np.log(p)-(1-y)*np.log1p(-p))),
        'Brier':float(np.dot(w,(y-p)**2)),'ECE15' if bins==15 and not equal_frequency else f'ECE{bins}_{"frequency" if equal_frequency else "width"}':float(ece),
        'Calibration_Intercept':intercept,'Calibration_Slope':slope,
        'Accuracy':float(np.dot(w,(p>=.5)==y)), 'NA_reason':'' if both else 'single_class'}

def make_hgb(iterations,l2,seed):
    return Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True)),('model',HistGradientBoostingClassifier(max_iter=iterations,learning_rate=.1,max_leaf_nodes=31,min_samples_leaf=20,l2_regularization=l2,random_state=seed,early_stopping=False))])

def fit_calibrator(name,p,y):
    p=safe_p(p)
    if name=='none': return {'name':'none'}
    if np.unique(y).size<2: raise ValueError('Calibration partition has a single class.')
    if name=='platt': return {'name':name,'model':LogisticRegression(C=1000,max_iter=500,random_state=20260713).fit(logit(p).reshape(-1,1),y)}
    result=minimize_scalar(lambda t:log_loss(y,expit(logit(p)/t),labels=[0,1]),bounds=(.5,2),method='bounded')
    return {'name':'temperature','temperature':float(result.x)}

def calibrate(obj,p):
    p=safe_p(p)
    if obj['name']=='none': return p
    if obj['name']=='platt': return safe_p(obj['model'].predict_proba(logit(p).reshape(-1,1))[:,1])
    return safe_p(expit(logit(p)/obj['temperature']))

def memory_fit(frame,y,p,column,shrinkage,cap):
    # Original NB05 one-step penalized logistic item-intercept update.
    tmp=pd.DataFrame({'key':frame[column].astype(str).to_numpy(),'residual':y-p,'hessian':p*(1-p)})
    agg=tmp.groupby('key').agg(residual=('residual','sum'),hessian=('hessian','sum'),support=('key','size'))
    agg['raw']=agg.residual/(agg.hessian+shrinkage);agg['correction']=agg['raw'].clip(-cap,cap)
    return {'delta':agg.correction.to_dict(),'support':agg.support.to_dict(),'raw':agg['raw'].to_dict(),'lambda':shrinkage,'cap':cap}

def hierarchy_fit(frame,p,qpar,fpar):
    y=frame.IsCorrect.to_numpy()
    return {name:memory_fit(frame,y,p,col,*par) for name,col,par in [('question','QuestionId',qpar),('subject','primary_subject_id',fpar),('parent','parent_subject_id',fpar)]}

def hierarchy_apply(frame,p,memory,variant='full'):
    delta=np.zeros(len(frame));route=np.full(len(frame),'zero',object);support=np.zeros(len(frame),int);lam=np.zeros(len(frame));clipped=np.zeros(len(frame),bool)
    for name,col in [('question','QuestionId'),('subject','primary_subject_id'),('parent','parent_subject_id')]:
        if (variant in ['no_question','hierarchy_only'] and name=='question') or (variant=='question_only' and name!='question') or (variant=='no_subject' and name=='subject'): continue
        m=memory[name];keys=frame[col].astype(str);values=keys.map(m['delta']);take=(route=='zero')&values.notna().to_numpy()
        delta[take]=values[take];route[take]=name;support[take]=keys[take].map(m['support']);lam[take]=m['lambda'];clipped[take]=keys[take].map(m['raw']).abs()>m['cap']
    return safe_p(expit(logit(safe_p(p))+delta)),pd.DataFrame({'route':route,'support_n':support,'shrinkage_lambda':lam,'clipped':clipped,'correction':delta})

def reuse_converged_helper():
    """Load only the existing pure helper, never the legacy experiment cells."""
    import nbformat
    n=nbformat.read(next((artifact_path('legacy/Notebooks')).glob('05_*.ipynb')),as_version=4)
    s=next(c.source for c in n.cells if c.id=='nested-fair-code')
    node=next(x for x in ast.parse(s).body if isinstance(x,ast.FunctionDef) and x.name=='fit_converged_ridge_item_intercepts')
    env={'np':np,'pd':pd,'logit':logit,'expit':expit,'safe_p':safe_p}
    exec(compile(ast.Module(body=[node],type_ignores=[]),'<existing NB05 Newton helper>','exec'),env)
    return env['fit_converged_ridge_item_intercepts']

def model_tests():
    z=np.array([-.9,.4,1.2]);y=np.array([0.,1.,1.]);lam=2.;d=.13;eps=1e-5
    objective=lambda t:float(np.logaddexp(0,z+t).sum()-y@(z+t)+lam*t*t/2)
    gradient=float((expit(z+d)-y).sum()+lam*d);hessian=float((expit(z+d)*(1-expit(z+d))).sum()+lam)
    assert np.isclose((objective(d+eps)-objective(d-eps))/(2*eps),gradient,rtol=1e-5)
    assert np.isclose((objective(d+eps)-2*objective(d)+objective(d-eps))/eps**2,hessian,rtol=1e-4)
    f=pd.DataFrame({'QuestionId':[1,1,1],'primary_subject_id':[1]*3,'parent_subject_id':[0]*3})
    expected=np.clip((y-expit(z)).sum()/((expit(z)*(1-expit(z))).sum()+lam),-.35,.35)
    # The fixture outcomes are explicit, not real experiment data.
    f['IsCorrect']=y
    memory=hierarchy_fit(f,expit(z),(lam,.35),(200.,.15));assert np.isclose(memory['question']['delta']['1'],expected)
    empty=f.iloc[:0];m=hierarchy_fit(empty,np.array([]),(2.,.35),(200.,.15));p,_=hierarchy_apply(f,expit(z),m);assert np.allclose(p,expit(z))
    return table('model_unit_tests.csv',pd.DataFrame([{'test':x,'status':'PASS','data':'synthetic_unit_test_only'} for x in ['gradient','hessian','first_newton_step','empty_memory_lookup']]),'NB03')

def temporal_parts(frame):
    dates=np.sort(frame.DateAnswered.unique());a=dates[int(.60*len(dates))];b=dates[int(.80*len(dates))]
    train=frame[frame.DateAnswered<a];memory=frame[(frame.DateAnswered>=a)&(frame.DateAnswered<b)];cal=frame[frame.DateAnswered>=b]
    for part in [train,memory,cal]:
        if part.IsCorrect.nunique()!=2: raise ValueError('Temporal partition lacks two classes; do not violate time to force fit.')
    assert train.DateAnswered.max()<memory.DateAnswered.min()<=memory.DateAnswered.max()<cal.DateAnswered.min()
    return train,memory,cal

def predict(est,frame,features=FEATURES): return safe_p(est.predict_proba(frame[features])[:,1])

def fit_candidates(frame,target,seed,features=FEATURES):
    train,mem,cal=temporal_parts(frame);full=pd.concat([train,mem],ignore_index=True);results={}
    for l2 in protocol()['hgb']['l2_candidates']:
        anchor=make_hgb(80,l2,seed);anchor.fit(train[features],train.IsCorrect)
        baseline80=make_hgb(80,l2,seed);baseline80.fit(full[features],full.IsCorrect)
        baseline100=make_hgb(100,l2,seed);baseline100.fit(full[features],full.IsCorrect)
        results[l2]={'anchor':anchor,'HGB80':baseline80,'HGB100':baseline100,'train':train,'mem':mem,'cal':cal,
            'anchor_mem':predict(anchor,mem,features),'anchor_cal':predict(anchor,cal,features),'anchor_target':predict(anchor,target,features),
            'HGB80_cal':predict(baseline80,cal,features),'HGB100_cal':predict(baseline100,cal,features),
            'HGB80_target':predict(baseline80,target,features),'HGB100_target':predict(baseline100,target,features)}
    return results

def select_inside(frame,seed,outer_fold):
    pr=protocol();dates=np.sort(frame.DateAnswered.unique());trials=[];memory_trials=[];buffers=[]
    for fold,(fi,hi) in enumerate(TimeSeriesSplit(n_splits=4).split(dates)):
        fit=frame[frame.DateAnswered.isin(dates[fi])];hold=frame[frame.DateAnswered.isin(dates[hi])]
        result=fit_candidates(fit,hold,seed+fold);buffers.append((hold,result))
        for l2,r in result.items():
            for name in ['HGB80','HGB100']:
                for cal_name in pr['calibration_candidates']:
                    cal=fit_calibrator(cal_name,r[name+'_cal'],r['cal'].IsCorrect)
                    p=calibrate(cal,r[name+'_target'])
                    trials.append({'model':name,'l2':l2,'calibration':cal_name,'inner_fold':fold,'Log_Loss':log_loss(hold.IsCorrect,p,labels=[0,1])})
            for qp in pr['memory_candidates']:
                for fp in pr['fallback_candidates']:
                    memory=hierarchy_fit(r['mem'],r['anchor_mem'],qp,fp)
                    p,_=hierarchy_apply(hold,r['anchor_target'],memory)
                    memory_trials.append({'l2':l2,'q_lambda':qp[0],'q_cap':qp[1],'f_lambda':fp[0],'f_cap':fp[1],'inner_fold':fold,'Log_Loss':log_loss(hold.IsCorrect,p,labels=[0,1])})
        print(f'seed={seed} outer={outer_fold} inner={fold+1}/4 completed',flush=True)
    selections={}
    for name in ['HGB80','HGB100']:
        scores=pd.DataFrame(trials);g=scores[scores.model.eq(name)].groupby(['l2','calibration']).Log_Loss.mean().sort_values();l2,c=g.index[0];selections[name]={'l2':float(l2),'calibration':c}
    scores=pd.DataFrame(memory_trials).groupby(['l2','q_lambda','q_cap','f_lambda','f_cap']).Log_Loss.mean().sort_values()
    l2,ql,qc,fl,fc=scores.index[0];cal_scores=[]
    for hold,result in buffers:
        r=result[l2];m=hierarchy_fit(r['mem'],r['anchor_mem'],(ql,qc),(fl,fc));pcal,_=hierarchy_apply(r['cal'],r['anchor_cal'],m);phold,_=hierarchy_apply(hold,r['anchor_target'],m)
        for name in pr['calibration_candidates']:
            cal=fit_calibrator(name,pcal,r['cal'].IsCorrect);cal_scores.append({'name':name,'Log_Loss':log_loss(hold.IsCorrect,calibrate(cal,phold),labels=[0,1])})
    c=pd.DataFrame(cal_scores).groupby('name').Log_Loss.mean().idxmin()
    selections['AdaptiveMath-AI']={'l2':float(l2),'question':[float(ql),float(qc)],'fallback':[float(fl),float(fc)],'calibration':c}
    ledger=pd.concat([pd.DataFrame(trials).assign(stage='HGB_tuning'),pd.DataFrame(memory_trials).assign(stage='memory_tuning')],ignore_index=True)
    ledger['seed']=seed;ledger['outer_fold']=outer_fold
    return selections,ledger

def prediction_frame(frame,p,model,seed,fold,kind,scenario,model_hash,route=None):
    out=frame[KEYS+['IsCorrect','DateAnswered','primary_subject_id','Gender','PremiumPupil','prior_interaction_count']].copy().reset_index(drop=True)
    out=out.rename(columns={'IsCorrect':'y'});out['p']=p;out['model']=model;out['seed']=seed;out['fold']=fold;out['scenario']=scenario;out['prediction_kind']=kind;out['model_hash']=model_hash
    if route is not None: out=pd.concat([out,route.reset_index(drop=True)],axis=1)
    return out

def fit_frozen(frame,target,selected,seed,fold,features=FEATURES):
    result=fit_candidates(frame,target,seed,features);payload={};predictions=[];fit_ids=[];convergence=[]
    for name,sel in selected.items():
        r=result[sel['l2']]
        if name=='AdaptiveMath-AI':
            m=hierarchy_fit(r['mem'],r['anchor_mem'],sel['question'],sel['fallback']);pc,_=hierarchy_apply(r['cal'],r['anchor_cal'],m);pt,route=hierarchy_apply(target,r['anchor_target'],m)
            cal=fit_calibrator(sel['calibration'],pc,r['cal'].IsCorrect)
            obj={'anchor':r['anchor'],'memory':m,'calibrator':cal,'features':features,'selected':sel,'seed':seed,'fold':fold}
            payload[name]=obj
            variants={'AdaptiveMath-AI':calibrate(cal,pt),'HGB80_anchor':r['anchor_target'],'Plus_question_memory':hierarchy_apply(target,r['anchor_target'],m,'question_only')[0],'Plus_hierarchy':pt,'No_calibration':pt}
            for variant in ['no_question','no_subject']:
                raw,rt=hierarchy_apply(target,r['anchor_target'],m,variant)
                # Preserve the identical frozen calibration mapping for a one-component contrast.
                variants['Ablation_'+variant]=calibrate(cal,raw)
            helper=reuse_converged_helper();mm,niter=helper(r['mem'].rename(columns={'QuestionId':'question_id'}),r['mem'].IsCorrect.to_numpy(),r['anchor_mem'],shrinkage=sel['question'][0])
            correction=target.QuestionId.astype(str).map(mm).fillna(0).clip(-sel['question'][1],sel['question'][1]).to_numpy()
            variants['Converged_item_intercept']=safe_p(expit(logit(r['anchor_target'])+correction))
            convergence.append({'seed':seed,'fold':fold,'iterations':niter,'converged':niter<50,'lambda':sel['question'][0],'cap':sel['question'][1],'helper':'existing NB05 fit_converged_ridge_item_intercepts'})
            for role,part in [('anchor',r['train']),('memory',r['mem']),('calibrator',r['cal'])]:
                fit_ids.append(part[KEYS+['DateAnswered']].assign(model=name,component=role,seed=seed,fold=fold))
        else:
            obj={'anchor':r[name],'memory':None,'calibrator':fit_calibrator(sel['calibration'],r[name+'_cal'],r['cal'].IsCorrect),'features':features,'selected':sel,'seed':seed,'fold':fold};payload[name]=obj
            variants={name:calibrate(obj['calibrator'],r[name+'_target'])};route=None
            for role,part in [('anchor',pd.concat([r['train'],r['mem']])),('calibrator',r['cal'])]:fit_ids.append(part[KEYS+['DateAnswered']].assign(model=name,component=role,seed=seed,fold=fold))
        path=artifact_path('Models')/f'{name.replace("-","_")}_seed{seed}_fold{fold}.joblib';joblib.dump(obj,path);mh=digest(path)
        restored=joblib.load(path)
        for variant,p in variants.items():
            if variant==name:
                pred=predict(restored['anchor'],target,features)
                if restored['memory'] is not None: pred,_=hierarchy_apply(target,pred,restored['memory'])
                np.testing.assert_allclose(calibrate(restored['calibrator'],pred),p,rtol=0,atol=1e-12)
            predictions.append(prediction_frame(target,p,variant,seed,fold,'temporal_outer','global_calendar_learner_feedback',mh,route if variant==name else None))
    return pd.concat(predictions,ignore_index=True),pd.concat(fit_ids,ignore_index=True),pd.DataFrame(convergence),payload

def primary_experiment():
    pr=protocol();frame=pd.read_parquet(artifact_path('Data/features_primary.parquet'));frame=frame[frame.split.ne('representation')].reset_index(drop=True)
    edges=pd.to_datetime(pr['calendar']['outer_edges'],utc=True)
    design={'budget':{'seeds':len(pr['seeds']),'outer':5,'inner':4},'outer_edges':list(map(str,edges)),
        'outer_fit_rule':'All supervised events before outer evaluation start; chronological 60/20/20 anchor/memory/calibration partitions.',
        'inner_rule':'4 expanding timestamp folds; each inner-fit is partitioned chronologically 60/20/20.',
        'opportunities':'Both HGB iteration budgets get identical 2 l2 candidates and 3 calibrators. Adaptive anchor gets same l2 candidates plus declared memory/fallback grids. Additional component tuning is disclosed, not described as equal candidate counts.',
        'fold_feedback':'Completed earlier outcomes are allowed learner feedback; item parameters fixed from representation window.',
        'original_grouped_analysis':'Legacy diagnostic retained in snapshot; not called global-time validation.'}
    path=artifact_path('temporal_design.json')
    if path.exists(): assert json.loads(path.read_text())==design
    else: save_json(path,design)
    inputs=[artifact_path('Data/features_primary.parquet'),artifact_path('Data/split_manifest.parquet'),artifact_path('revision_protocol.json'),artifact_path('feature_contract.json')]
    base_hash={str(p.relative_to(V)):digest(p) for p in inputs}
    base_hash['revision_models.py']=code_digest(__name__)
    summaries=[]
    with threadpool_limits(limits=4):
        for seed in pr['seeds']:
            for fold,(start,end) in enumerate(zip(edges[:-1],edges[1:])):
                out=artifact_path('Data')/f'predictions_seed{seed}_fold{fold}.parquet';meta=artifact_path('manifests')/f'fit_seed{seed}_fold{fold}.json'
                contract=dict(input_code_helper_feature_split_hashes=base_hash,data_helper_hash=code_digest('revision_data'),seed=seed,fold=fold,start=str(start),end=str(end))
                if out.exists() and meta.exists():
                    m=json.loads(meta.read_text())
                    if fit_contract_matches(m['contract'],contract) and m['prediction_hash']==digest(out) and all((artifact_path(p)).exists() and digest(artifact_path(p))==h for p,h in m['models'].items()):
                        print(f'Validated exact cache seed={seed} fold={fold}',flush=True);summaries+=m['summary'];continue
                    raise RuntimeError('Cache mismatch: dependent chain must be recomputed in a new execution version.')
                fit=frame[frame.DateAnswered<start];target=frame[(frame.DateAnswered>=start)&(frame.DateAnswered<end)]
                if target.empty: raise ValueError('Empty temporal evaluation window; cannot force old fold count.')
                selected,ledger=select_inside(fit,seed,fold)
                preds,ids,conv,payload=fit_frozen(fit,target,selected,seed,fold)
                preds.to_parquet(out,index=False);ids.to_parquet(artifact_path('Data')/f'fitting_ids_seed{seed}_fold{fold}.parquet',index=False)
                ledger.to_parquet(artifact_path('Tables')/f'tuning_seed{seed}_fold{fold}.parquet',index=False);conv.to_csv(artifact_path('Tables')/f'convergence_seed{seed}_fold{fold}.csv',index=False)
                current=[]
                for model,g in preds.groupby('model'):
                    current.append({'seed':seed,'fold':fold,'model':model,'N':len(g),**metrics(g.y,g.p)})
                mh={str(p.relative_to(V)):digest(p) for p in (artifact_path('Models')).glob(f'*seed{seed}_fold{fold}.joblib')}
                save_json(meta,{'contract':contract,'selected':selected,'models':mh,'calibration_hashes':{k:joblib.hash(v['calibrator']) for k,v in payload.items()},'prediction_hash':digest(out),'summary':current})
                summaries+=current;table('primary_per_seed_fold_metrics.csv',pd.DataFrame(summaries),'NB05')
                print(f'Completed seed={seed} outer={fold+1}/5; targets={len(target)}',flush=True)
    return pd.DataFrame(summaries).groupby('model')[['ROC_AUC','Log_Loss','Brier','ECE15']].agg(['mean','std'])

In [2]:
import revision_models as rm
display(rm.model_tests())

,test,status,data
0,gradient,PASS,synthetic_unit_test_only
1,hessian,PASS,synthetic_unit_test_only
2,first_newton_step,PASS,synthetic_unit_test_only
3,empty_memory_lookup,PASS,synthetic_unit_test_only


In [3]:
import revision_sensitivity as rs
display(rs.psychometric_checks())

,model,fit_rows,fit_max_time,iterations,converged,identifiability,ROC_AUC,AP_correct,AP_incorrect,Log_Loss,Brier,ECE15,Calibration_Intercept,Calibration_Slope,Accuracy,NA_reason
0,Joint_ridge_logistic_student_item_intercepts,180585,2019-05-31 20:01:00+00:00,283.0,True,L2 penalty identifies student/item contributio...,0.637140,0.753885,0.480025,0.627158,0.217849,0.035430,0.143714,0.670363,0.659485,
1,BKT_fixed,180585,NaN,NaN,None,NaN,0.637749,0.756568,0.456502,0.674748,0.238291,0.141143,0.426702,0.412135,0.596371,
2,BKT_train_fitted,180585,NaN,NaN,True,NaN,0.667741,0.770161,0.496602,0.608042,0.210010,0.012120,-0.018020,1.062976,0.671494,
3,BKT_prespecified_sensitivity,180585,NaN,NaN,None,NaN,0.638967,0.756014,0.459083,0.669680,0.238395,0.133029,0.497819,0.483363,0.598837,
